# Demand Forecast — Error Analysis

Run the training pipeline first. This notebook compares the selected forecast with the seasonal baseline and examines error concentration.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ARTIFACTS = Path("../artifacts")
forecasts = pd.read_csv(ARTIFACTS / "test_forecasts.csv", parse_dates=["timestamp"])
metrics = json.loads((ARTIFACTS / "metrics.json").read_text())
metrics


In [ ]:
tail = forecasts.tail(24 * 14)
figure, axis = plt.subplots(figsize=(13, 5))
axis.plot(tail["timestamp"], tail["actual"], label="Actual", color="#0F172A")
axis.plot(tail["timestamp"], tail["selected_forecast"], label="Selected forecast", color="#2563EB")
axis.set(title="Latest Two Weeks of Held-out Demand", ylabel="Demand")
axis.legend()
axis.grid(alpha=0.25)
figure.autofmt_xdate()
figure.tight_layout()


In [ ]:
forecasts["absolute_error"] = (forecasts["actual"] - forecasts["selected_forecast"]).abs()
forecasts["hour"] = forecasts["timestamp"].dt.hour
hourly = forecasts.groupby("hour", as_index=False)["absolute_error"].mean()
figure, axis = plt.subplots(figsize=(10, 5))
axis.bar(hourly["hour"], hourly["absolute_error"], color="#7C3AED")
axis.set(title="Mean Absolute Error by Hour", xlabel="Hour", ylabel="MAE")
figure.tight_layout()
